# Desarrollo del modelo de predicción de precios Airbnb CDMX

Este notebook documenta el proceso completo utilizado para construir el modelo de predicción de precios por noche para alojamientos de Airbnb en Ciudad de México.

**Objetivo:** estimar `price` en MXN a partir de características del alojamiento, anfitrión, disponibilidad, ubicación y tipo de propiedad.

**Resultado principal:** el mejor desempeño lo obtuvo la regresión lineal múltiple, con `R2_test = 0.6293` y `RMSE_test = 376.42`.


## Cómo leer este notebook

El código está organizado como una receta. Cada sección responde una pregunta:

- **¿Qué datos tengo?** Se carga `listings.csv`.
- **¿Qué debo limpiar?** Se transforma `price`, porcentajes y booleanos.
- **¿Qué voy a predecir?** La variable objetivo es `price`.
- **¿Con qué variables voy a predecir?** `X` contiene las variables explicativas y `y` contiene el precio.
- **¿Cómo entreno?** Se arma un `Pipeline` para que la limpieza, codificación y modelo viajen juntos.
- **¿Cómo sé si funciona?** Se comparan métricas en entrenamiento y prueba.

Los nombres más importantes son:

- `df_original`: datos sin tocar.
- `df`: datos después de la limpieza básica.
- `df_limpio`: datos ya filtrados sin outliers de precio.
- `X`: columnas usadas para predecir.
- `y`: precio real que el modelo intenta aprender.
- `m_lineal`: pipeline final de regresión lineal múltiple.


## Diagrama de flujo del proceso

Este diagrama resume el flujo completo desde la carga del dataset hasta la integraci?n del modelo en Flask.

```mermaid
flowchart TD
    A["Inicio: listings.csv"] --> B["Carga de datos con pandas"]
    B --> C["Limpieza inicial"]
    C --> C1["Eliminar URLs, IDs, texto libre y columnas administrativas"]
    C --> C2["Convertir price a número"]
    C --> C3["Convertir porcentajes y booleanos"]
    C1 --> D["Filtrar precios válidos"]
    C2 --> D
    C3 --> D
    D --> E["Eliminar outliers con regla IQR"]
    E --> F["Seleccionar variables predictoras"]
    F --> F1["Variables numéricas"]
    F --> F2["Variables categóricas"]
    F1 --> G["Pipeline numérico: mediana + escalado"]
    F2 --> H["Pipeline categórico: moda + OneHotEncoder"]
    G --> I["ColumnTransformer"]
    H --> I
    I --> J["Separación train/test 80/20"]
    J --> K1["Regresión lineal múltiple"]
    J --> K2["Regresión polinómica grado 2"]
    J --> K3["Regresión polinómica grado 3"]
    K1 --> L["Evaluar MAE, RMSE y R²"]
    K2 --> L
    K3 --> L
    L --> M["Seleccionar mejor modelo"]
    M --> N["Guardar modelo_lineal.pkl"]
    N --> O["Integrar en Flask"]
    O --> P["Dashboard y calculadora de precio"]
```


## 1. Configuración e importación de librerías

Se usan `pandas` y `numpy` para preparar datos, `scikit-learn` para el preprocesamiento y entrenamiento, y `joblib` para guardar el modelo final.


In [ ]:
from pathlib import Path

import joblib

import numpy as np

import pandas as pd

from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", "{:,.4f}".format)

ROOT = Path(".")
DATA_PATH = ROOT / "listings.csv"
TARGET = "price"

PLOTS_DIR = ROOT / "static" / "plots"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)


## 2. Carga de datos

El dataset base es `listings.csv`, proveniente de Inside Airbnb para Ciudad de México. Incluye 27,051 registros y 79 columnas antes de la limpieza.


In [ ]:
df_original = pd.read_csv(DATA_PATH, low_memory=False)

print(f"Filas y columnas originales: {df_original.shape}")

df_original.head()


## 3. Limpieza inicial

Se eliminaron columnas que no aportaban al modelo predictivo o que eran texto libre, URLs, identificadores, fechas o campos altamente administrativos.

También se normalizaron tres tipos de campos:

- `price`: se removieron símbolos de moneda y comas para convertir a número.
- Tasas porcentuales: `host_response_rate` y `host_acceptance_rate` pasaron de texto con `%` a número.
- Booleanos: variables con `t`/`f` se transformaron a `1`/`0`.


In [ ]:
df = df_original.copy()

cols_to_drop = [
    "id", "listing_url", "scrape_id", "last_scraped", "source",
    "name", "description", "neighborhood_overview", "picture_url",
    "host_id", "host_url", "host_name", "host_since", "host_location",
    "host_about", "host_thumbnail_url", "host_picture_url",
    "host_neighbourhood", "host_verifications", "host_has_profile_pic",
    "host_identity_verified", "neighbourhood", "neighbourhood_group_cleansed",
    "bathrooms_text", "calendar_updated", "last_review", "license",
    "calculated_host_listings_count_entire_homes",
    "calculated_host_listings_count_private_rooms",
    "calculated_host_listings_count_shared_rooms",
]

df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

df[TARGET] = df[TARGET].astype(str).str.replace(r"[\$,]", "", regex=True)
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")

for col in ["host_response_rate", "host_acceptance_rate"]:
    if col in df.columns:
        df[col] = pd.to_numeric(
            df[col].astype(str).str.replace("%", "").str.strip(),
            errors="coerce",
        )

for col in ["host_is_superhost", "has_availability", "instant_bookable"]:
    if col in df.columns:
        df[col] = df[col].map({"t": 1, "f": 0, True: 1, False: 0}).astype(float)

print(f"Columnas despues de eliminar campos no usados: {df.shape[1]}")

df[[TARGET]].describe()


## 4. Tratamiento de nulos y outliers en la variable objetivo

Primero se eliminaron precios nulos o no positivos. Después se aplicó la regla IQR para retirar valores extremos de `price`, porque los precios atípicos distorsionan la regresión y aumentan el error.


In [ ]:
df = df.dropna(subset=[TARGET]).copy()

df = df[df[TARGET] > 0].copy()

q1 = df[TARGET].quantile(0.25)
q3 = df[TARGET].quantile(0.75)
iqr = q3 - q1

lower_limit = q1 - 1.5 * iqr
upper_limit = q3 + 1.5 * iqr

df_limpio = df[(df[TARGET] >= lower_limit) & (df[TARGET] <= upper_limit)].copy()

print(f"Registros con precio valido: {len(df):,}")
print(f"Registros despues de IQR: {len(df_limpio):,}")
print(f"Limites IQR: {lower_limit:,.2f} a {upper_limit:,.2f} MXN")
df_limpio[TARGET].describe()


## 5. Selección de variables

Para el modelo lineal se conservaron todas las variables numéricas disponibles y cuatro variables categóricas clave:

- `host_response_time`
- `neighbourhood_cleansed`
- `property_type`
- `room_type`

Para los modelos polinómicos se usaron solo las 10 variables numéricas con mayor correlación absoluta con `price`, evitando una explosión de memoria al generar terminos polinómicos.


In [ ]:
useful_cat = [
    "host_response_time",
    "neighbourhood_cleansed",
    "property_type",
    "room_type",
]

X = df_limpio.drop(columns=[TARGET])

categorical_features = [c for c in useful_cat if c in X.columns]

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

X = X[numeric_features + categorical_features]

y = df_limpio[TARGET]

corr = df_limpio[numeric_features + [TARGET]].corr(numeric_only=True)[TARGET].drop(TARGET).abs()
top10_features = corr.nlargest(10).index.tolist()

print(f"Features numericas: {len(numeric_features)}")
print(f"Features categoricas: {len(categorical_features)}")

pd.DataFrame({
    "feature": top10_features,
    "abs_corr_price": corr.loc[top10_features].values,
})


## 6. Preprocesamiento

Se construyó un `ColumnTransformer` para manejar numéricas y categóricas de forma consistente dentro del pipeline:

- Numéricas: imputación por mediana y estandarización.
- Categóricas: imputación por moda y `OneHotEncoder` con `handle_unknown="ignore"` para tolerar categorias nuevas.


In [ ]:
num_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

cat_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("num", num_transformer, numeric_features),
    ("cat", cat_transformer, categorical_features),
])

def evaluar(nombre, modelo, Xtr, Xte, ytr, yte):

    ytr_pred = modelo.predict(Xtr)

    yte_pred = modelo.predict(Xte)

    return {
        "Modelo": nombre,

        "MAE_train": round(mean_absolute_error(ytr, ytr_pred), 2),

        "RMSE_train": round(np.sqrt(mean_squared_error(ytr, ytr_pred)), 2),

        "R2_train": round(r2_score(ytr, ytr_pred), 4),
        "MAE_test": round(mean_absolute_error(yte, yte_pred), 2),
        "RMSE_test": round(np.sqrt(mean_squared_error(yte, yte_pred)), 2),
        "R2_test": round(r2_score(yte, yte_pred), 4),
    }, yte_pred


## 7. Separacion entrenamiento/prueba

Se uso una partición 80/20 con `random_state=42` para asegurar reproducibilidad.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

Xn = df_limpio[top10_features].copy()
yn = df_limpio[TARGET]

Xn_train, Xn_test, yn_train, yn_test = train_test_split(
    Xn, yn, test_size=0.20, random_state=42
)

print(f"Train lineal: {X_train.shape}, Test lineal: {X_test.shape}")
print(f"Train polinomico: {Xn_train.shape}, Test polinomico: {Xn_test.shape}")


## 8. Entrenamiento de modelos

Se compararon tres alternativas:

- Regresión lineal múltiple con numéricas + categóricas codificadas.
- Regresión polinómica grado 2 usando top 10 numéricas.
- Regresión polinómica grado 3 usando top 10 numéricas.


In [ ]:
m_lineal = Pipeline([
    ("preprocessor", preprocessor),
    ("reg", LinearRegression()),
])

m_lineal.fit(X_train, y_train)

res_lin, pred_lin = evaluar(
    "Regresion lineal multiple",
    m_lineal,
    X_train,
    X_test,
    y_train,
    y_test,
)

m_poly2 = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("scl", StandardScaler()),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("reg", LinearRegression()),
])
m_poly2.fit(Xn_train, yn_train)
res_p2, pred_p2 = evaluar(
    "Regresion polinomica grado 2",
    m_poly2,
    Xn_train,
    Xn_test,
    yn_train,
    yn_test,
)

m_poly3 = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("scl", StandardScaler()),
    ("poly", PolynomialFeatures(degree=3, include_bias=False)),
    ("reg", LinearRegression()),
])
m_poly3.fit(Xn_train, yn_train)
res_p3, pred_p3 = evaluar(
    "Regresion polinomica grado 3",
    m_poly3,
    Xn_train,
    Xn_test,
    yn_train,
    yn_test,
)


## 9. Comparación de desempeño

El modelo lineal múltiple fue seleccionado porque logro el mejor equilibrio entre error, capacidad explicativa y estabilidad fuera de muestra.


In [ ]:
resultados = pd.DataFrame([res_lin, res_p2, res_p3])

resultados.sort_values("R2_test", ascending=False)


## 10. Interpretación del modelo lineal

Para explicar el modelo final se extrajeron coeficientes después del `OneHotEncoder`. Los coeficientes positivos aumentan la predicción y los negativos la reducen, considerando que las numéricas fueron estandarizadas.


In [ ]:
ohe = m_lineal.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"]
cat_names = list(ohe.get_feature_names_out(categorical_features))

feature_names = [
    (f[:40] + "...") if len(f) > 40 else f
    for f in numeric_features + cat_names
]

coefs = np.ravel(m_lineal.named_steps["reg"].coef_)

coef_df = pd.DataFrame({
    "Variable": feature_names[: len(coefs)],
    "Coeficiente": coefs,
}).sort_values("Coeficiente", ascending=False)

display(coef_df.head(10))
display(coef_df.tail(10))


## 11. Exportación de artefactos

Estos archivos alimentan el dashboard Flask y permiten reutilizar el modelo sin reentrenar:

- `modelo_lineal.pkl`: pipeline final entrenado.
- `resultados_modelos.csv`: comparativa de métricas.
- `coeficientes_modelo_lineal.csv`: interpretabilidad del modelo lineal.
- `template_prediccion.csv`: plantilla de entrada para predicciones.
- `scatter_lineal.csv`, `scatter_poly2.csv`, `correlación.csv`, `price_by_neighbourhood.csv`, `price_sample.csv`: datos para visualizaciones.


In [ ]:
EXPORTAR_ARTEFACTOS = True

if EXPORTAR_ARTEFACTOS:
    resultados.to_csv("resultados_modelos.csv", index=False)

    coef_df.to_csv("coeficientes_modelo_lineal.csv", index=False)

    joblib.dump(m_lineal, "modelo_lineal.pkl")

    X_train.iloc[[0]].to_csv("template_prediccion.csv", index=False)

    scatter_lin = pd.DataFrame({"real": y_test.values, "pred": pred_lin})
    scatter_lin.sample(min(2000, len(scatter_lin)), random_state=42).to_csv(
        "scatter_lineal.csv", index=False
    )

    scatter_p2 = pd.DataFrame({"real": yn_test.values, "pred": pred_p2})
    scatter_p2.sample(min(2000, len(scatter_p2)), random_state=42).to_csv(
        "scatter_poly2.csv", index=False
    )

    corr_export = (
        df_limpio[numeric_features + [TARGET]]
        .corr(numeric_only=True)[TARGET]
        .drop(TARGET)
        .sort_values(ascending=False)
    )
    corr_export.reset_index().rename(
        columns={"index": "feature", TARGET: "corr"}
    ).to_csv("correlacion.csv", index=False)

    price_by_neighbourhood = (
        df_limpio.groupby("neighbourhood_cleansed")[TARGET]
        .median()
        .sort_values(ascending=False)
        .reset_index()
    )
    price_by_neighbourhood.columns = ["neighbourhood", "median_price"]
    price_by_neighbourhood.to_csv("price_by_neighbourhood.csv", index=False)

    df_limpio[[TARGET]].sample(min(3000, len(df_limpio)), random_state=42).to_csv(
        "price_sample.csv", index=False
    )

print("Artefactos exportados.")


## 12. Prueba de predicción

La aplicación Flask carga `modelo_lineal.pkl` y usa `template_prediccion.csv` como base para recibir datos del formulario, reemplazar las variables seleccionadas y devolver una predicción.


In [ ]:
modelo_final = joblib.load("modelo_lineal.pkl")

ejemplo = pd.read_csv("template_prediccion.csv")

for col, value in {
    "neighbourhood_cleansed": "Cuauhtémoc",
    "room_type": "Entire home/apt",
    "property_type": "Entire rental unit",
    "host_response_time": "within an hour",
    "accommodates": 2,
    "bedrooms": 1,
    "beds": 1,
    "bathrooms": 1,
    "minimum_nights": 2,
    "availability_365": 180,
}.items():
    if col in ejemplo.columns:
        ejemplo[col] = value

prediccion = modelo_final.predict(ejemplo)[0]

print(f"Precio estimado: ${max(prediccion, 0):,.2f} MXN por noche")


## 13. Conclusiones

- La regresión lineal múltiple fue el modelo elegido por obtener el mejor `R2_test` y el menor `RMSE_test`.
- Las variables de capacidad del alojamiento, tipo de propiedad, tipo de cuarto y ubicación explican buena parte de la variación del precio.
- Los modelos polinómicos no mejoraron el resultado; el grado 3 mostro peor generalizacion.
- El resultado final se integró en un dashboard con visualizaciones, mapa y calculadora de precio en tiempo real.

**Mejoras futuras:** validar con datos más recientes, incorporar variables de amenities en forma estructurada y probar modelos no lineales regularizados como Random Forest, Gradient Boosting o Elastic Net.
